In [5]:
# Step 1 : conda create --name pro python==3.10
# Step 2 : conda activate pro
# Step 3 : pip install ipykernel
# Step 4 : python -m ipykernel install --user --name pro --display-name pro
# pip install kagglehub

In [16]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import tree
from sklearn.preprocessing import (LabelEncoder, 
                                   StandardScaler, 
                                   MinMaxScaler)

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (confusion_matrix, 
                             accuracy_score,
                             classification_report)

In [14]:
# # Download latest version
# path = kagglehub.dataset_download("ruchikakumbhar/coupon-acception-prediction")
# print("Path to dataset files:", path)

In [17]:
df = pd.read_csv("coupon.csv")
print(df.shape)
df.head()

(12684, 25)


,destination,passanger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,...,Bar,CoffeeHouse,CarryAway,Restaurant20,Restaurant20To50,dist_15,dist_25,direction_same,direction_opp,Y
0,No Urgent Place,Alone,Sunny,55,2PM,Restaurant(<20),1d,Female,21,Unmarried partner,...,never,never,NaN,4~8,1~3,0,0,0,1,1
1,No Urgent Place,Friend(s),Sunny,80,10AM,Coffee House,2h,Female,21,Unmarried partner,...,never,never,NaN,4~8,1~3,0,0,0,1,0
2,No Urgent Place,Friend(s),Sunny,80,10AM,Carry out & Take away,2h,Female,21,Unmarried partner,...,never,never,NaN,4~8,1~3,1,0,0,1,1
3,No Urgent Place,Friend(s),Sunny,80,2PM,Coffee House,2h,Female,21,Unmarried partner,...,never,never,NaN,4~8,1~3,1,0,0,1,0
4,No Urgent Place,Friend(s),Sunny,80,2PM,Coffee House,1d,Female,21,Unmarried partner,...,never,never,NaN,4~8,1~3,1,0,0,1,0


In [18]:
df.isnull().sum()

destination             0
passanger               0
weather                 0
temperature             0
time                    0
coupon                  0
expiration              0
gender                  0
age                     0
maritalStatus           0
children                0
education               0
occupation              0
income                  0
car                 12576
Bar                   107
CoffeeHouse           217
CarryAway             151
Restaurant20          130
Restaurant20To50      189
dist_15                 0
dist_25                 0
direction_same          0
direction_opp           0
Y                       0
dtype: int64

In [19]:
df = df.drop("car", axis=1)

miss_data = ["Bar", "CoffeeHouse", "CarryAway", "Restaurant20", "Restaurant20To50"]

for col in miss_data:
    mode = df[col].mode().iloc[0]
    df[col].fillna(mode, inplace=True)
    
df.isnull().sum()

C:\Users\Dev\AppData\Local\Temp\ipykernel_3248\1107442157.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(mode, inplace=True)


destination         0
passanger           0
weather             0
temperature         0
time                0
coupon              0
expiration          0
gender              0
age                 0
maritalStatus       0
children            0
education           0
occupation          0
income              0
Bar                 0
CoffeeHouse         0
CarryAway           0
Restaurant20        0
Restaurant20To50    0
dist_15             0
dist_25             0
direction_same      0
direction_opp       0
Y                   0
dtype: int64

In [20]:
# check category level of all columns
for col in df.columns:
    print("----------------{}-----------------------".format(col))
    print(df[col].value_counts())
    print()

----------------destination-----------------------
destination
No Urgent Place    6283
Home               3237
Work               3164
Name: count, dtype: int64

----------------passanger-----------------------
passanger
Alone        7305
Friend(s)    3298
Partner      1075
Kid(s)       1006
Name: count, dtype: int64

----------------weather-----------------------
weather
Sunny    10069
Snowy     1405
Rainy     1210
Name: count, dtype: int64

----------------temperature-----------------------
temperature
80    6528
55    3840
30    2316
Name: count, dtype: int64

----------------time-----------------------
time
6PM     3230
7AM     3164
10AM    2275
2PM     2009
10PM    2006
Name: count, dtype: int64

----------------coupon-----------------------
coupon
Coffee House             3996
Restaurant(<20)          2786
Carry out & Take away    2393
Bar                      2017
Restaurant(20-50)        1492
Name: count, dtype: int64

----------------expiration-----------------------
expiratio

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12684 entries, 0 to 12683
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   destination       12684 non-null  object
 1   passanger         12684 non-null  object
 2   weather           12684 non-null  object
 3   temperature       12684 non-null  int64 
 4   time              12684 non-null  object
 5   coupon            12684 non-null  object
 6   expiration        12684 non-null  object
 7   gender            12684 non-null  object
 8   age               12684 non-null  object
 9   maritalStatus     12684 non-null  object
 10  children          12684 non-null  int64 
 11  education         12684 non-null  object
 12  occupation        12684 non-null  object
 13  income            12684 non-null  object
 14  Bar               12684 non-null  object
 15  CoffeeHouse       12684 non-null  object
 16  CarryAway         12684 non-null  object
 17  Restaurant20

In [23]:
# apply label encoder to convert categories into numeric values
le = LabelEncoder()

for x in df.columns:
    
    if df[x].dtypes == "object":
        print(x)
        df[x]=le.fit_transform(df[x])
        le_name = dict(zip(le.classes_, le.transform(le.classes_)))
        print("--------------------------------------------------")
        print("Feature: ", x)
        print("Mapping: ", le_name)  
    

destination
--------------------------------------------------
Feature:  destination
Mapping:  {'Home': np.int64(0), 'No Urgent Place': np.int64(1), 'Work': np.int64(2)}
passanger
--------------------------------------------------
Feature:  passanger
Mapping:  {'Alone': np.int64(0), 'Friend(s)': np.int64(1), 'Kid(s)': np.int64(2), 'Partner': np.int64(3)}
weather
--------------------------------------------------
Feature:  weather
Mapping:  {'Rainy': np.int64(0), 'Snowy': np.int64(1), 'Sunny': np.int64(2)}
time
--------------------------------------------------
Feature:  time
Mapping:  {'10AM': np.int64(0), '10PM': np.int64(1), '2PM': np.int64(2), '6PM': np.int64(3), '7AM': np.int64(4)}
coupon
--------------------------------------------------
Feature:  coupon
Mapping:  {'Bar': np.int64(0), 'Carry out & Take away': np.int64(1), 'Coffee House': np.int64(2), 'Restaurant(20-50)': np.int64(3), 'Restaurant(<20)': np.int64(4)}
expiration
--------------------------------------------------
Feat

In [24]:
df.head()

,destination,passanger,weather,temperature,time,coupon,expiration,gender,age,maritalStatus,...,Bar,CoffeeHouse,CarryAway,Restaurant20,Restaurant20To50,dist_15,dist_25,direction_same,direction_opp,Y
0,1,0,2,55,2,4,0,0,0,3,...,4,4,0,1,0,0,0,0,1,1
1,1,1,2,80,0,2,1,0,0,3,...,4,4,0,1,0,0,0,0,1,0
2,1,1,2,80,0,1,1,0,0,3,...,4,4,0,1,0,1,0,0,1,1
3,1,1,2,80,2,2,1,0,0,3,...,4,4,0,1,0,1,0,0,1,0
4,1,1,2,80,2,2,0,0,0,3,...,4,4,0,1,0,1,0,0,1,0


In [27]:
X = df.iloc[  :  ,  :-1]  # independent variables
y = df.iloc[  :  ,   -1] 

# Split the data into test and train
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.3,
                                                    random_state=10)  
# verify spliting data
print("Total Data: ", df.shape)
print("Train X: ", X_train.shape)
print("Train y: ", y_train.shape)
print("Test X: ", X_test.shape)
print("Test y: ", y_test.shape)

# perform data normalisation

# create object for scaler
scaler = MinMaxScaler()

# fit the train data on scaler object 
scaler.fit(X_train)

X_train_scale = scaler.transform(X_train)
X_test_scale = scaler.transform(X_test)

Total Data:  (12684, 24)
Train X:  (8878, 23)
Train y:  (8878,)
Test X:  (3806, 23)
Test y:  (3806,)


In [38]:
from sklearn.ensemble import (BaggingClassifier, 
                              RandomForestClassifier, 
                              AdaBoostClassifier, 
                              GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
# from catboost import catboost

In [39]:
def build_ensembel_model(X_train, X_test, y_train, y_test):
    
    model_list = []
    
    m1 = BaggingClassifier()
    m2 = RandomForestClassifier()
    m3 = AdaBoostClassifier()
    m4 = GradientBoostingClassifier()
    m5 = DecisionTreeClassifier()
    m6 = LogisticRegression()
    m7 = SVC()
#     m8 = catboost()
    
    model_list.append(("Bagging Model", m1))
    model_list.append(("RandomForest Model", m2))
    model_list.append(("Adaboost Model", m3))
    model_list.append(("GradientBoost Model", m4))
    model_list.append(("DecisionTree Model", m5))
    model_list.append(("LogisticRegression Model", m6))
    model_list.append(("Svc Model", m7))
#     model_list.append(("Catboost", m8))
    
    results = []
    for name, model in model_list:
        print("Training {} ...............".format(name))    
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        
        print("---------Confusion Matrix------------")
        print(confusion_matrix(y_test, y_pred))
        print()
        print("---------Accuracy Score------------")
        print(accuracy_score(y_test,y_pred))
        print()
        print("---------Classification Report------------")
        print(classification_report(y_test,y_pred))

        print("Model Training Complete............")
        print()

In [40]:
build_ensembel_model(X_train_scale, X_test_scale, y_train, y_test)

Training Bagging Model ...............
---------Confusion Matrix------------
[[1146  536]
 [ 563 1561]]

---------Accuracy Score------------
0.711245401996847

---------Classification Report------------
              precision    recall  f1-score   support

           0       0.67      0.68      0.68      1682
           1       0.74      0.73      0.74      2124

    accuracy                           0.71      3806
   macro avg       0.71      0.71      0.71      3806
weighted avg       0.71      0.71      0.71      3806

Model Training Complete............

Training RandomForest Model ...............
---------Confusion Matrix------------
[[1106  576]
 [ 381 1743]]

---------Accuracy Score------------
0.7485549132947977

---------Classification Report------------
              precision    recall  f1-score   support

           0       0.74      0.66      0.70      1682
           1       0.75      0.82      0.78      2124

    accuracy                           0.75      3806
   mac

## Lazypredict 

In [41]:
from lazypredict.Supervised import LazyClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [42]:
clf = LazyClassifier(verbose=1,ignore_warnings=True, custom_metric=None)
models,predictions = clf.fit(X_train_scale, X_test_scale, y_train, y_test)


  3%|██▌                                                                                | 1/32 [00:00<00:10,  2.95it/s]

{'Model': 'AdaBoostClassifier', 'Accuracy': 0.6791907514450867, 'Balanced Accuracy': np.float64(0.6652475194314006), 'ROC AUC': np.float64(0.6652475194314006), 'F1 Score': 0.6738431989909099, 'Time taken': 0.33746767044067383}


 12%|██████████▍                                                                        | 4/32 [00:00<00:04,  5.76it/s]

{'Model': 'BaggingClassifier', 'Accuracy': 0.7233315817130846, 'Balanced Accuracy': np.float64(0.7208792106966193), 'ROC AUC': np.float64(0.7208792106966193), 'F1 Score': 0.7236629132727188, 'Time taken': 0.30785393714904785}
{'Model': 'BernoulliNB', 'Accuracy': 0.6192853389385181, 'Balanced Accuracy': np.float64(0.599821752867965), 'ROC AUC': np.float64(0.599821752867965), 'F1 Score': 0.6076918927396615, 'Time taken': 0.034064531326293945}
{'Model': 'CalibratedClassifierCV', 'Accuracy': 0.6211245401996847, 'Balanced Accuracy': np.float64(0.5998612202762831), 'ROC AUC': np.float64(0.5998612202762831), 'F1 Score': 0.6071476982780873, 'Time taken': 0.1360323429107666}


 22%|██████████████████▏                                                                | 7/32 [00:00<00:02, 10.26it/s]

...

{'Model': 'RandomForestClassifier', 'Accuracy': 0.7409353652128219, 'Balanced Accuracy': np.float64(0.7314550765723704), 'ROC AUC': np.float64(0.7314550765723703), 'F1 Score': 0.738696218919852, 'Time taken': 1.009314775466919}
{'Model': 'RidgeClassifier', 'Accuracy': 0.6200735680504467, 'Balanced Accuracy': np.float64(0.5994144828034064), 'ROC AUC': np.float64(0.5994144828034064), 'F1 Score': 0.60691414974728, 'Time taken': 0.03071451187133789}
{'Model': 'RidgeClassifierCV', 'Accuracy': 0.6203363110877562, 'Balanced Accuracy': np.float64(0.5996498876998282), 'ROC AUC': np.float64(0.5996498876998282), 'F1 Score': 0.6071438760547752, 'Time taken': 0.043799400329589844}
{'Model': 'SGDClassifier', 'Accuracy': 0.581187598528639, 'Balanced Accuracy': np.float64(0.5650075799816827), 'ROC AUC': np.float64(0.5650075799816827), 'F1 Score': 0.5732309923626915, 'Time taken': 0.07938694953918457}


 88%|███████████████████████████████████████████████████████████████████████▊          | 28/32 [00:29<00:05,  1.30s/it]

{'Model': 'SVC', 'Accuracy': 0.6873357856016815, 'Balanced Accuracy': np.float64(0.6707511235615389), 'ROC AUC': np.float64(0.670751123561539), 'F1 Score': 0.6797501813146067, 'Time taken': 5.446794509887695}


100%|██████████████████████████████████████████████████████████████████████████████████| 32/32 [00:30<00:00,  1.05it/s]


[LightGBM] [Info] Number of positive: 5086, number of negative: 3792
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000885 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 138
[LightGBM] [Info] Number of data points in the train set: 8878, number of used features: 23
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.572877 -> initscore=0.293598
[LightGBM] [Info] Start training from score 0.293598
{'Model': 'LGBMClassifier', 'Accuracy': 0.7477666841828692, 'Balanced Accuracy': np.float64(0.7370807217665276), 'ROC AUC': np.float64(0.7370807217665276), 'F1 Score': 0.744922558312742, 'Time taken': 0.17917990684509277}


In [43]:
models

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
LGBMClassifier,0.75,0.74,0.74,0.74,0.18
RandomForestClassifier,0.74,0.73,0.73,0.74,1.01
BaggingClassifier,0.72,0.72,0.72,0.72,0.31
ExtraTreesClassifier,0.73,0.72,0.72,0.73,1.47
NuSVC,0.70,0.69,0.69,0.70,6.56
DecisionTreeClassifier,0.68,0.68,0.68,0.68,0.07
SVC,0.69,0.67,0.67,0.68,5.45
AdaBoostClassifier,0.68,0.67,0.67,0.67,0.34
ExtraTreeClassifier,0.66,0.65,0.65,0.66,0.04


In [44]:
predictions

,Accuracy,Balanced Accuracy,ROC AUC,F1 Score,Time Taken
Model,,,,,
LGBMClassifier,0.75,0.74,0.74,0.74,0.18
RandomForestClassifier,0.74,0.73,0.73,0.74,1.01
BaggingClassifier,0.72,0.72,0.72,0.72,0.31
ExtraTreesClassifier,0.73,0.72,0.72,0.73,1.47
NuSVC,0.70,0.69,0.69,0.70,6.56
DecisionTreeClassifier,0.68,0.68,0.68,0.68,0.07
SVC,0.69,0.67,0.67,0.68,5.45
AdaBoostClassifier,0.68,0.67,0.67,0.67,0.34
ExtraTreeClassifier,0.66,0.65,0.65,0.66,0.04
